In [1]:
import pandas as pd
import numpy as np 
from pathlib import Path
from sentence_transformers import SentenceTransformer

In [2]:
BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"

songs = pd.read_csv(PROCESSED_DIR / "song_master.csv")

songs_with_lyrics = songs[songs["lyrics_clean"].notna().copy()]

print(songs.shape)
print(songs_with_lyrics.shape)

(441, 18)
(381, 18)


In [5]:
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
lyrics_texts = songs_with_lyrics["lyrics_clean"].tolist()

embeddings = model.encode(
    lyrics_texts,
    show_progress_bar=True, 
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings.shape

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

(381, 384)

In [8]:
embedding_cols = [f"embedding_{i}" for i in range(embeddings.shape[1])]

embeddings_df = pd.DataFrame(
    embeddings,
    columns=embedding_cols
)

song_embeddings = pd.concat(
    [
        songs_with_lyrics[[
            "track_id", 
            "track_name",
            "album_name",
            "release_year"
        ]].reset_index(drop=True),
        embeddings_df
    ],
    axis=1
)

song_embeddings.head()

,track_id,track_name,album_name,release_year,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,...,embedding_374,embedding_375,embedding_376,embedding_377,embedding_378,embedding_379,embedding_380,embedding_381,embedding_382,embedding_383
0,6nt3AoYjkaqXMZhypTBky1,SWIM,KEEP SWIMMING,2026,0.039356,-0.029155,0.127304,0.057019,-0.050284,-0.003185,...,0.081385,0.101313,-0.081610,-0.012466,-0.125231,0.112798,0.003728,-0.011990,-0.018128,-0.016811
1,7EytKcb3klVPpN5IW1sj1Y,SWIM with RM (Chill Hip Hop Remix),KEEP SWIMMING,2026,0.041490,-0.028483,0.128478,0.059912,-0.051265,-0.007168,...,0.086504,0.098872,-0.079756,-0.009005,-0.122758,0.110451,0.000700,-0.009140,-0.022843,-0.018910
2,5dZLsPskKzph16LWo31uxL,SWIM with Jin (Alternative Rock Remix),KEEP SWIMMING,2026,0.039885,-0.028030,0.128368,0.056939,-0.048508,-0.001793,...,0.080903,0.102320,-0.080652,-0.011518,-0.124695,0.111934,0.004748,-0.010834,-0.016463,-0.016273
3,5AL5OrvyIMPqKjl9iw3xO5,SWIM with SUGA (Melodic Techno Remix),KEEP SWIMMING,2026,0.039356,-0.029155,0.127304,0.057019,-0.050284,-0.003185,...,0.081385,0.101313,-0.081610,-0.012466,-0.125231,0.112798,0.003728,-0.011990,-0.018128,-0.016811
4,3MCJY7lXCHa0UNIjsAucaJ,SWIM with j-hope (Afrobeat Remix),KEEP SWIMMING,2026,0.039885,-0.028030,0.128368,0.056939,-0.048508,-0.001793,...,0.080903,0.102320,-0.080652,-0.011518,-0.124695,0.111934,0.004748,-0.010834,-0.016463,-0.016273


In [10]:
song_embeddings.to_parquet(
    PROCESSED_DIR / "song_embeddings.parquet",
    index=False
)

print("Saved:", PROCESSED_DIR / "song_embeddings.parquet")
print("Shape:", song_embeddings.shape)

Saved: ../data/processed/song_embeddings.parquet
Shape: (381, 388)


In [11]:
check = pd.read_parquet(PROCESSED_DIR / "song_embeddings.parquet")

print(check.shape)
check.head()

(381, 388)


,track_id,track_name,album_name,release_year,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,...,embedding_374,embedding_375,embedding_376,embedding_377,embedding_378,embedding_379,embedding_380,embedding_381,embedding_382,embedding_383
0,6nt3AoYjkaqXMZhypTBky1,SWIM,KEEP SWIMMING,2026,0.039356,-0.029155,0.127304,0.057019,-0.050284,-0.003185,...,0.081385,0.101313,-0.081610,-0.012466,-0.125231,0.112798,0.003728,-0.011990,-0.018128,-0.016811
1,7EytKcb3klVPpN5IW1sj1Y,SWIM with RM (Chill Hip Hop Remix),KEEP SWIMMING,2026,0.041490,-0.028483,0.128478,0.059912,-0.051265,-0.007168,...,0.086504,0.098872,-0.079756,-0.009005,-0.122758,0.110451,0.000700,-0.009140,-0.022843,-0.018910
2,5dZLsPskKzph16LWo31uxL,SWIM with Jin (Alternative Rock Remix),KEEP SWIMMING,2026,0.039885,-0.028030,0.128368,0.056939,-0.048508,-0.001793,...,0.080903,0.102320,-0.080652,-0.011518,-0.124695,0.111934,0.004748,-0.010834,-0.016463,-0.016273
3,5AL5OrvyIMPqKjl9iw3xO5,SWIM with SUGA (Melodic Techno Remix),KEEP SWIMMING,2026,0.039356,-0.029155,0.127304,0.057019,-0.050284,-0.003185,...,0.081385,0.101313,-0.081610,-0.012466,-0.125231,0.112798,0.003728,-0.011990,-0.018128,-0.016811
4,3MCJY7lXCHa0UNIjsAucaJ,SWIM with j-hope (Afrobeat Remix),KEEP SWIMMING,2026,0.039885,-0.028030,0.128368,0.056939,-0.048508,-0.001793,...,0.080903,0.102320,-0.080652,-0.011518,-0.124695,0.111934,0.004748,-0.010834,-0.016463,-0.016273


In [12]:
song_embeddings.isna().sum().sum()

np.int64(0)

In [13]:
len([c for c in song_embeddings.columns if c.startswith("embedding_")])


384

In [15]:
song_embeddings["track_id"].is_unique

True

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

black_swan = song_embeddings[
    song_embeddings["track_name"] == "Black Swan"
].iloc[0]

fake_love = song_embeddings[
    song_embeddings["track_name"] == "FAKE LOVE"
].iloc[0]

tomorrow = song_embeddings[
    song_embeddings["track_name"] == "Tomorrow"
].iloc[0]




In [18]:
embedding_cols = [c for c in song_embeddings.columns if c.startswith("embedding_")]

black_vec = black_swan[embedding_cols].values.reshape(1, -1)
fake_vec = fake_love[embedding_cols].values.reshape(1, -1)
tomorrow_vec = tomorrow[embedding_cols].values.reshape(1, -1)


In [19]:
print(
    "Black Swan ↔ Fake Love:",
    cosine_similarity(black_vec, fake_vec)[0][0]
)

print(
    "Black Swan ↔ Tomorrow:",
    cosine_similarity(black_vec, tomorrow_vec)[0][0]
)

Black Swan ↔ Fake Love: 0.2817943261021951
Black Swan ↔ Tomorrow: 0.42539363401625996


## Top 10 msot similar songs to Black Swan

In [21]:
embedding_cols = [c for c in song_embeddings.columns if c.startswith("embedding_")]

target_song = "Tomorrow"

target_vector = song_embeddings.loc[
    song_embeddings["track_name"] == target_song, 
    embedding_cols
].values

similarities = cosine_similarity(
    target_vector, 
    song_embeddings[embedding_cols]
)[0]

results = song_embeddings[[
    "track_name",
    "album_name", 
    "release_year"
]].copy()

results["similarity"] = similarities

results = results.sort_values(
    "similarity", 
    ascending=False
)

results.head(10)

,track_name,album_name,release_year,similarity
291,Tomorrow,Skool Luv Affair (Special Addition),2014,1.000000
300,Tomorrow,Skool Luv Affair,2014,1.000000
78,Life Goes On,BE,2020,0.615705
59,Life Goes On,Proof,2022,0.615705
81,Telepathy,BE,2020,0.590538
244,Run (Alternative Mix),The Most Beautiful Moment in Life: Young Forever,2016,0.586329
283,So 4 more,Dark & Wild,2014,0.581902
247,Run,The Most Beautiful Moment in Life Pt.2,2015,0.570724
228,Run,The Most Beautiful Moment in Life: Young Forever,2016,0.570724
49,RUN,Proof,2022,0.570724
